---   
 <img align="left" width="75" height="75"  src="https://upload.wikimedia.org/wikipedia/en/c/c8/University_of_the_Punjab_logo.png"> 

<h1 align="center">Department of Data Science</h1>

---
<h3><div align="right">Instructor: Muhammad Arif Butt, Ph.D.</div></h3>    

<br><br>
<h1 align="center">Lec-48: Deploy an AI Agent on Render and Access it from Python</h1>

# Learning agenda of this notebook

1. **Recap: From an Agent in a Notebook to an Agent Service**
    - The agent loop so far ran inside our notebook (Lec-41, Lec-45); today it runs on a server
    - Architecture: Python client → HTTPS → agent service on Render → LLM (Groq) + remote MCP servers
2. **The Agent We Deploy**
    - The MCP-powered agent from Lec-45 (section 6b), switched to **remote** MCP servers (our Demo-server and the Booking server)
    - Why not the coding agent: shell commands, local files and `input()` approvals do not belong on a public server
3. **Turning the Agent into a Web Service with FastAPI**
    - From the `input()` chat loop to an HTTP endpoint: `POST /chat`
    - Where the conversation lives: per-session memory on the server vs. history sent by the client
    - Connecting to the MCP servers once, at start-up
    - A `/health` route, and protecting the endpoint with a bearer token
4. **Setting Up the Project with uv and git**
    - Project layout, dependencies, `.env` and `.gitignore`
5. **Running and Testing the Agent Service Locally**
    - Start the server, then test it with `curl`, the interactive `/docs` page, and Python
6. **Deploying the Agent on Render**
    - Push to GitHub, create a Web Service, set environment variables and the health check
    - Living with the free plan: spin-down, cold starts and lost memory
7. **Accessing the Deployed Agent from Python**
    - One request with `requests`
    - A chat loop in Python that talks to the agent in the cloud
    - Several users at once with an async client (`httpx`)
8. **Making the Agent Service Production-Ready**
    - Guardrails, timeouts, rate limits and cost, logging and secrets
9. **Security and Troubleshooting**
10. **Key Takeaways and Check Your Understanding**

> **Decided (for later writing):** this lecture deploys the **MCP-powered agent from Lec-45, section 6b** (the Lec-41 agent loop with tools from MCP servers), switched to the remote Demo-server (`https://mcp-render-server-xzbh.onrender.com/mcp`) and the Booking server (`https://booking-mcp-server.onrender.com/mcp`). The Lec-45 coding agent (capstone, section 6c) is **not** deployed, for the reasons given in section 2.

# <span style='background :lightgreen' >1. Recap: From an Agent in a Notebook to an Agent Service</span>

> **To be written.** Planned content:
- **Where we are:** in Lec-41 and Lec-45 the agent loop (`run_tool`, `run_agent`, `main`) ran inside our notebook, and only we could use it. A deployed agent runs on a server, so **any program** (a Python script, a web page, a mobile app) can send it a message and get an answer.
- **Agent = Model + Harness (Lec-39):** today we move the **harness** to a server. The model still runs at the LLM provider, and the tools still run on the MCP servers.
- **Architecture figure** (new image, in the `agent-new-*` style): Python client → HTTPS → **agent service on Render** (FastAPI + the agent loop) → Groq LLM, and → remote MCP servers (Demo-server, Booking server).
- **Compare with Lec-44:** there we deployed an **MCP server** (tools only, no LLM); here we deploy an **agent** (an LLM that decides which tools to call). A table: what each one is, who calls it, and what it calls.

# <span style='background :lightgreen' >2. The Agent We Deploy</span>

> **To be written.** Planned content:
- **Starting point:** the MCP-powered agent from **Lec-45, section 6b**: the Lec-41 agent loop with tools from an MCP server.
- **One change:** instead of starting the Demo-server locally over stdio, it connects to **remote** MCP servers, because a cloud server cannot start programs from our laptop:
    - our Demo-server on Render: `https://mcp-render-server-xzbh.onrender.com/mcp`
    - the Booking server: `https://booking-mcp-server.onrender.com/mcp`
- **Run it once in the notebook** (a short recap cell) so students see the agent working before it is deployed.
- **Why not the Lec-45 coding agent?** It runs shell commands, reads and writes local files, and asks for approval with `input()`. On a public server there is no one to type `y`, and any user could run commands on the machine.

# <span style='background :lightgreen' >3. Turning the Agent into a Web Service with FastAPI</span>

> **To be written.** Planned content:
- **From `input()` to HTTP:** the `while True: input()` loop is replaced by an endpoint. The client sends `POST /chat` with `{"session_id": "...", "message": "..."}` and gets back `{"reply": "...", "tool_calls": [...]}`. `run_tool()` and `run_agent()` stay exactly as in Lec-45.
- **Where the conversation (`messages`) lives**, compared in a table:
    - **Option A, server-side memory:** a dictionary `{session_id: messages}` on the server. Simple, but it is lost when the free Render instance restarts or spins down.
    - **Option B, client-side history:** the client sends the whole history with each request, and the server stays **stateless**. More robust, but every request is bigger.
- **Connect to the MCP servers once** when the service starts (FastAPI's `lifespan`), not on every request.
- **`GET /health`** for Render's health check and for waking the service.
- **Protect `/chat` with a bearer token** (`AGENT_API_TOKEN`), the same idea as `MCP_API_TOKEN` in Lec-44: without it, anyone who finds the URL can spend our Groq quota.
- **Guardrail:** add the `MAX_STEPS` limit from Lec-39 to `run_agent()`, because a server must never loop forever.

# <span style='background :lightgreen' >4. Setting Up the Project with uv and git</span>

> **To be written.** Planned content:
- Create the project with `uv init --no-package agent_service` (plain `uv init` now creates a `src/` package with no `main.py`).
- `uv add fastapi uvicorn openai fastmcp python-dotenv`
- Files: `main.py` (the service), `.env` (local keys, never committed), `.gitignore`, `pyproject.toml`, `uv.lock`.
- `main.py` reads the port from the `PORT` environment variable and listens on `0.0.0.0`, as Render requires (Lec-44).

# <span style='background :lightgreen' >5. Running and Testing the Agent Service Locally</span>

> **To be written.** Planned content:
- Start it with `uv run main.py` and open `http://127.0.0.1:8000/health`.
- Try the agent from FastAPI's interactive **`/docs`** page (Swagger UI).
- Call it with `curl`, including the `Authorization: Bearer ...` header, and see a request without the token rejected with `401`.
- Call it from a notebook cell with `requests`: the same question as Lec-45 (*"Is it warmer in Lahore or in Islamabad right now?"*), now answered by the local service.

# <span style='background :lightgreen' >6. Deploying the Agent on Render</span>

> **To be written.** Planned content:
- Step by step, following the same pattern as Lec-44: push the project to **GitHub** → **New › Web Service** on Render → Build Command `uv sync --frozen && uv cache prune --ci`, Start Command `uv run main.py`, Instance Type **Free**.
- **Environment variables:** `GROQ_API_KEY`, `AGENT_API_TOKEN`, and the MCP server URLs. Keys live on Render, never in the code or on the students' machines.
- Health check path `/health`; reading the **Logs** tab to watch the agent's tool calls.
- **The free plan:** the service sleeps after 15 minutes without traffic, the first request takes about a minute, and server-side memory (option A) is lost when it sleeps. Our agent also calls two other free Render services, so a cold start can wake up to three services in a row.

# <span style='background :lightgreen' >7. Accessing the Deployed Agent from Python</span>

> **To be written.** Planned content:
- **a. One request** with `requests.post(...)`: URL, token header, JSON body, and a generous timeout for cold starts.
- **b. A chat loop in Python:** the familiar `while True: input("You: ")` loop from Lec-41 and Lec-45, but each turn is an HTTP request to the agent in the cloud, with a `session_id` so the agent remembers the conversation.
- **c. Several users at once:** an async client with `httpx` sends questions from several sessions in parallel, and each one gets its own conversation.
- **Compare** what runs where: the client (only HTTP), the agent service (the loop), the LLM provider (the model), the MCP servers (the tools).

# <span style='background :lightgreen' >8. Making the Agent Service Production-Ready</span>

> **To be written.** Planned content:
- **Guardrails:** `MAX_STEPS`, a timeout on every LLM and MCP call, and a maximum message length.
- **Errors:** catch LLM and MCP errors (for example Groq's `429` rate limit) and return a clear error response instead of crashing the service.
- **Cost and rate limits:** every request uses the deployer's Groq quota; limit requests per token or per IP.
- **Observability:** log each request, each tool call and its duration, and read them in Render's Logs tab.
- **Optional:** streaming the reply as it is generated (Server-Sent Events), and where frameworks such as the OpenAI Agents SDK fit in.

# <span style='background :lightgreen' >9. Security and Troubleshooting</span>

> **To be written.** Planned content:
- **Security:** the bearer token, keys only in environment variables, never trusting the user's message (prompt injection can make the agent call tools), and giving the deployed agent only safe, read-only tools.
- **Troubleshooting table** (symptom → likely cause → fix): cold-start timeouts, `401 Unauthorized`, *"No open ports detected"* on deploy, `uv sync --frozen` failures, MCP server unreachable from Render, Groq `429` / `413` errors, lost conversations after a spin-down.

# <span style='background :lightgreen' >10. Key Takeaways and Check Your Understanding</span>

> **To be written.** Planned content:
- **Key takeaways:** a deployed agent is the same agent loop behind an HTTP endpoint; the client only sends messages; the server owns the loop, the keys and the guardrails; and where the conversation is stored decides whether the service can be stateless.
- **Check your understanding:** about 10 questions covering the architecture, the `/chat` endpoint, the two ways to store the conversation, the bearer token, the free plan's limits, and the difference between deploying an MCP server (Lec-44) and deploying an agent (this lecture).